## Taxo-Tag

This notebook demonstrates the use of taxo-tag.



First import the packages used.

In [3]:
import pandas as pd
import networkx as nx
from sentence_transformers import SentenceTransformer

import taxo

Then fill a pandas dataframe with a taxonomy, for instance by loading it from an excel sheet.

Each row corresponds to a taxonomy concept. Each concept has to have a key, a label_en, a description_en (optionally also label and description in other languages), and a parents attribute, which is a string that interprets as the list of its parent concepts (note that not only trees but also more general DAGs are allowed as taxonomic hierarchies).

There has to be on special formal concept which is the top parent of all other concepts.

In [4]:
df_media_tax = pd.read_excel("media.xlsx")
df_media_tax.fillna("", inplace=True)

The example contained in the file `media.xlsx` is extracted from the IPTC media topics taxonomy, a taxonomy often used for subject classification of news articles.

Note that the top parent node has the key "M00000000".

In the following, we use the embedding model [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

In [5]:
media_tax = taxo.ConceptTaxonomy(nx.DiGraph(), "M00000000", transformer=SentenceTransformer("all-MiniLM-L6-v2"))
media_tax.from_df(df_media_tax)

Now, given an example text ...

In [6]:
text = """
AI opens up new possibilities for air traffic control and the cockpit.

How will artificial intelligence (AI) support air traffic controllers and pilots in the future? Over the past four years, 
the German Aerospace Center (Deutsches Zentrum für Luft- und Raumfahrt; DLR) has been investigating this question as part 
of the LOKI project (Kollaboration von Luftfahrt-OperateurInnen und KI-Systemen, or 'Collaboration between Aviation 
Operators and AI Systems' in English). In collaboration with national and international companies and research 
institutions from the air traffic control sector, as well as public authorities, the project also developed relevant 
recommendations for introducing AI into aviation.

The aviation industry is currently working intensively on introducing AI systems. Unlike conventional assistance systems, 
these can process large volumes of data, be trained on the basis of this data and so adapt flexibly to new situations. 
AI systems offer the potential to increase the efficiency of air traffic and relieve the workload on highly qualified 
specialists, while at the same time ensuring safety, transparency and human control at all times.

"In the DLR LOKI project, we investigated how humans and AI systems can cooperate successfully in aviation. Our aim was to 
make this collaboration trustworthy, transparent and user-centred," says the project’s scientific lead, Carmen Bruder from 
the DLR Institute of Aerospace Medicine.

The researchers developed a digital air traffic controller as an AI-supported team member for air traffic control and an 
intelligent assistance system for pilots, and made recommendations for the safe introduction of trustworthy AI into aviation.

One focus of the research was the development of the digital air traffic controller DIRC (Digital Interactive Reliable 
Controller). Unlike traditional assistance systems, the DIRC not only supports human controllers with information or 
recommendations but can also take on certain tasks independently. This allows for dynamic coordination with human air traffic 
controllers regarding which tasks are to be undertaken.

Simulations showed that human and digital air traffic controllers can divide their tasks efficiently. Together, they managed 
up to 25 percent more than the current maximum traffic volume specified for an airspace sector. The research thus addresses 
a key challenge facing European air traffic: rising traffic volumes coupled with the increasing challenge of maintaining 
staffing levels in air traffic control.

In addition to air traffic control, the researchers developed the Intelligent Pilot Assistance System (IPAS), a demonstrator 
designed to support pilots in the cockpit.

The system is designed to support crews particularly in time-critical situations, for example when selecting alternative 
flight routes or suitable alternate airports. The AI analyses large amounts of relevant information and processes it to aid 
decision-making. At the same time, the team investigated what information pilots need in order to understand and reliably 
assess the recommendations of an AI system.

In addition to technical developments, the project investigated the conditions necessary for trustworthy collaboration 
between humans and AI systems. The researchers analysed the expectations and requirements of aviation personnel and assessed 
how collaboration between humans and AI in aviation should be structured.

A key finding is that trust is not built solely on a system’s performance. Crucial factors include transparent information 
about the AI's capabilities and limitations, suitable training opportunities and the gradual introduction of new functions. 
Based on the project findings, important recommendations were formulated; taking these into account will help ensure the 
successful development and introduction of AI systems in aviation.

The findings form a basis for the further development of AI systems in aviation and will feed into the European debate on the 
safe introduction of such systems.
"""

... we can prepare it for tagging.

In [7]:
taxed_document = taxo.TaxedDocument(media_tax, text)

The following set of hyperparameters typically works well with this taxonomy. Other taxonomies might require tuning of these parameters.

In [8]:
tagging_parameters = (
    10,  # Max number of concepts initially assigned to each sentence
    0.25,  # Minimum similarity value on sentence level
    2,  # Weight for title sentence
    1.5,  # Saturation parameter for score aggregation
    0.25,  # Minimum value for aggregated score 
    80,  # Percentage for hierarchy truncation on top level
    80,  # Percentage for hierarchy truncation on second level
    7,  # Maximum number of returned leaf concepts 
    2,  # Number of paths to the root considered per concept
)

Now we are ready to identify the taxonomy tags for the given text. The result shows the highest scored concepts including their hierarchical anchestry.

In [9]:
taxed_document.identify_concepts(*tagging_parameters, with_scores=True)

[[('economy, business and finance', 'M04000000', 0.572107398702342),
  ('economic sector', 'M20000209', 0.572107398702342),
  ('transport', 'M20000337', 0.572107398702342),
  ('air transport', 'M20000338', 0.572107398702342)],
 [('science and technology', 'M13000000', 0.4709945980217274),
  ('technology and engineering', 'M20000756', 0.4709945980217274),
  ('information technology and computer science',
   'M20000763',
   0.4709945980217274),
  ('artificial intelligence', 'M20001298', 0.4709945980217274)],
 [('disaster, accident and emergency incident',
   'M03000000',
   0.4322114299130465),
  ('accident and emergency incident', 'M20000139', 0.4322114299130465),
  ('transportation accident and incident', 'M20000143', 0.4322114299130465),
  ('air and space accident and incident', 'M20000144', 0.4322114299130465)],
 [('science and technology', 'M13000000', 0.4314064954667891),
  ('technology and engineering', 'M20000756', 0.4314064954667891),
  ('aerospace engineering', 'M20000757', 0.4